In [17]:
import numpy as np

seed = 1234
np.random.seed(seed=seed)

In [18]:
# para entrenar un quantum support vector machine necesitamos datos clasicos
# se toman estos datos de sklearn.datasets
from sklearn.datasets import load_wine
x,y = load_wine(return_X_y=True) # return_X_y=True para que también devuelva las etiquetas (y)

In [19]:
categorias = []
[categorias.append(elemento) for elemento in y if elemento not in categorias]
print(categorias)
print('---------------------------------------------------------------------')
print(y)

[0, 1, 2]
---------------------------------------------------------------------
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [20]:
# la base de datos incluye tres categorias distintas, pero solo 3
# queremos dos categorias que clasificar

# segun la documentacion de esta base de datos: los primeros 59 elementos
# pertenecen a la primera categoria (etiqueta 0), los 71 siguientes pertencen a la segunda
# categoria (etiqueta 1) y el resto de elementos a la tercera categoria (etiqueta 2)
# tomamos unicamente los 59+71= 130 primeros elementos

x = x[:59+71]
y = y[:59+71]

In [21]:
# separamos la base de datos en base de datos de entrenamiento y prueba
# no creamos base de datos de validacion 
from sklearn.model_selection import train_test_split
x_tr, x_test, y_tr, y_test = train_test_split(x,y,train_size=0.9)

In [22]:
# la mayoria de los mapeos caracteristicos requieren datos normalizados
# aunque no fuera el caso, la normalizacion de los datos es una buena practica en machine learning
# los normalizaremos tal que el valor absoluto maximo que tomen los valores sea 1
from sklearn.preprocessing import MaxAbsScaler
scaler = MaxAbsScaler()
x_tr = scaler.fit_transform(x_tr)

In [23]:
print(x_tr)
print(len(x_tr[0]))

[[0.92447741 0.32068966 0.73065015 ... 0.64912281 1.         0.61607143]
 [0.84086312 0.26206897 0.68111455 ... 0.67836257 0.6575     0.5577381 ]
 [0.94538098 0.28965517 0.68421053 ... 0.60818713 0.8975     0.61607143]
 ...
 [0.92852326 0.32758621 0.82972136 ... 0.66081871 0.7325     0.81845238]
 [0.88132165 0.25862069 0.6501548  ... 0.69005848 0.6725     0.60714286]
 [0.93728928 0.28965517 0.65634675 ... 0.53216374 0.8325     0.58630952]]
13


Todos las variables eran positivas, por lo que todos los valores en la base de entrenamiento oscilan entre 0 y 1.
Si existieran variables negativas oscilarian entre -1 y 1 una vez normalizadas

In [24]:
# se normaliza la base de datos de prueba tambien
x_test = scaler.transform(x_test)
x_test = np.clip(x_test,0,1)

In [25]:
import pennylane as qml

In [26]:
# un quantum support vector machine es un support vector machine pero con un kernel cuantico
# se implementa ese kernel cuantico en pennylane
# la base de datos tiene 13 variables -> necesitariamos 13 qubits si queremos implementar ZZ feature map
# o 4 qubits si queremos implementar amplitude enconding
nqubits = 4
dev = qml.device('default.qubit', wires = nqubits)

la funcion kernel en QSVM se define de la siguiente forma $k(\vec{a}, \vec{b}) = \big | \langle 0 \big | \Psi^* (\vec b) \Psi (\vec a) \big | 0 \rangle  \big | ^ 2 $, que es la probabilidad de obtener 0 al medir el estado $ \Psi^* (\vec b) \Psi (\vec a) \big | 0 \rangle  $

In [27]:
# se define el circuito de la funcion kernel con amplitude encoding como feature mapping

@qml.qnode(dev)
def kernel_circ(a,b): # a y b son los datos clasicos que el kernel toma como inputs para hallar su producto escalar en el espacio caracteristico
    # en QSVM el espacio caracteristico es el espacio de estados cuanticos
    qml.AmplitudeEmbedding(
        a, wires=range(nqubits), pad_with=0, # pad_with=0 establece como 0 los valores de los qubits que no se empleen
        normalize=True) # normalize=True normaliza los inputs
    qml.adjoint(qml.AmplitudeEmbedding(
        b, wires=range(nqubits), pad_with=0, normalize=True
    ))
    return qml.probs(wires=range(nqubits))


In [28]:
# ahora hemos de usar esta funcion kernel en un support vector machine
from sklearn.svm import SVC
def qkernel(A,B):
    # kernel_circ(a,b)[0]: devuelve la probabilidad de medir |0> al aplicar el circuito kernel a |0>
    return np.array([[kernel_circ(a,b)[0] for b in B] for a in A])

svm = SVC(kernel=qkernel).fit(x_tr, y_tr)

In [29]:
# comprobamos la precision del modelo
from sklearn.metrics import accuracy_score
print(accuracy_score(y_pred=svm.predict(x_test), y_true=y_test))

0.9230769230769231


Hemos usado amplitude encoding como feature mapping dado que la base de datos incluye 13 variables. Para usar angle enconding o zz feature tendriamos que crear circuitos de 13 qubits en principio, pero podemos reducir la dimensionalidad de la base de datos con analisis de componente principal

In [30]:
from sklearn.decomposition import PCA

pca = PCA(n_components=8)
xs_tr = pca.fit_transform(x_tr)
xs_test = pca.transform(x_test)

In [31]:
nqubits = 8
dev = qml.device('default.qubit', wires=nqubits)

@qml.qnode(dev)
def kernel_circ_angle_embdedding(a,b):
    qml.AngleEmbedding(a, wires=range(nqubits))
    qml.adjoint(qml.AngleEmbedding(b, wires=range(nqubits)))
    return qml.probs(wires=range(nqubits))

In [32]:
def qkernel_angle_embedding(A,B):
    # kernel_circ(a,b)[0]: devuelve la probabilidad de medir |0> al aplicar el circuito kernel a |0>
    return np.array([[kernel_circ_angle_embdedding(a,b)[0] for b in B] for a in A])

svm = SVC(kernel = qkernel_angle_embedding).fit(xs_tr, y_tr)

In [33]:
print(accuracy_score(y_pred=svm.predict(xs_test), y_true=y_test))

1.0


La precision devuelta es de 1 -> clasificacion perfecta para este caso

Podemos implementar nuestros propios mapeos caracteristicos ademas de los que se incluyen en las librerias

In [34]:
from itertools import combinations

# escribimos ZZ Feature Map 
def ZZFeatureMap(nqubits, data):
    nload = min(len(data), nqubits) # numero de variables

    for i in range(nload):
        qml.Hadamard(i)
        qml.RZ(2.0 * data[i], wires=i)
    
    for pair in list(combinations(range(nload),2)):
        q0 = pair[0]
        q1 = pair[1]

        qml.CZ(wires=[q0, q1])
        qml.RZ(2.0 * (np.pi - data[q0])*(np.pi - data[q1]),wires=q1)
        qml.CZ(wires=[q0,q1]) 

In [35]:
nqubits = 4
dev = qml.device('default.qubit', wires=nqubits)

@qml.qnode(dev)
def kernel_circ_ZZ(a,b):
    ZZFeatureMap(nqubits,a)
    qml.adjoint(ZZFeatureMap)(nqubits,b)
    return qml.probs(wires=range(nqubits))

def qkernel_ZZ(A,B):
    # kernel_circ(a,b)[0]: devuelve la probabilidad de medir |0> al aplicar el circuito kernel a |0>
    return np.array([[kernel_circ_ZZ(a,b)[0] for b in B] for a in A])

svm = SVC(kernel=qkernel_ZZ).fit(xs_tr,y_tr)

In [36]:
print(accuracy_score(y_pred=svm.predict(xs_test),y_true=y_test))

0.8461538461538461
